In [ ]:
!pip install evaluate prettytable peft

In [ ]:
!pip install -U torchao

ERROR: Operation cancelled by user


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
# point this at whatever folder contains train.py
sys.path.insert(0, "/content/drive/MyDrive/bgf_v2")

import os
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import json

# pull in the custom classes
from train import ESM2LoRAForSequenceClassification, ESMConfig


class TokenizedDataset(Dataset):
    def __init__(self, csv_file, tokenizer, label_mapper):
        self.data = pd.read_csv(csv_file)
        self.tokenizer = tokenizer
        self.label_mapper = label_mapper

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sequence = self.data.iloc[idx]['sequence']
        label = self.label_mapper[self.data.iloc[idx]['cycle']]
        inputs = self.tokenizer(sequence)
        return {
            'input_ids':      inputs['input_ids'],
            'attention_mask': inputs['attention_mask'],
            'labels':         label,
        }

Mounted at /content/drive


In [ ]:
class ModelWithTemperature(nn.Module):
    def __init__(self, model):
        super(ModelWithTemperature, self).__init__()
        self.model = model
        # parametrize as log_temperature so T = exp(log_T) is always > 0
        self.log_temperature = nn.Parameter(torch.log(torch.tensor(1.5)))

    @property
    def temperature(self):
        return self.log_temperature.exp()

    def forward(self, input_ids):
        logits = self.model(input_ids).logits
        return self.temperature_scale(logits)

    def temperature_scale(self, logits):
        return logits / self.temperature

    def reset_temperature(self, init=1.0):
        with torch.no_grad():
            self.log_temperature.fill_(float(np.log(init)))

    def fit_temperature(self, logits, labels, save_path,
                        device="cuda:0" if torch.cuda.is_available() else "cpu",
                        output_dir=None,
                        before_nll=None, before_ece=None):
        """
        Fit temperature on PRE-COMPUTED logits/labels. Caller is responsible
        for collecting them once.
        """
        self.to(device)
        logits = logits.to(device)
        labels = labels.to(device)
        nll_criterion = nn.CrossEntropyLoss().to(device)

        if before_nll is None:
            before_nll = nll_criterion(logits, labels).item()
        if before_ece is None:
            before_ece = calculate_ece(logits, labels)
        print(f"Before temperature - NLL: {before_nll:.3f}, ECE: {before_ece:.3f}")

        optimizer = optim.LBFGS([self.log_temperature], lr=0.01, max_iter=50)

        def closure():
            optimizer.zero_grad()
            scaled = self.temperature_scale(logits)
            loss = nll_criterion(scaled, labels)
            loss.backward()
            return loss

        print("Optimizing temperature (NLL only)...")
        optimizer.step(closure)

        after_logits = self.temperature_scale(logits).detach()
        after_nll = nll_criterion(after_logits, labels).item()
        after_ece = calculate_ece(after_logits, labels)
        T_final = self.temperature.item()

        print(f"Optimal temperature: {T_final:.3f}")
        print(f"After temperature - NLL: {after_nll:.3f}, ECE: {after_ece:.3f}")

        if output_dir:
            make_reliability_diagram(
                after_logits, labels,
                title=f"After Temperature Scaling\nECE: {after_ece:.3f}",
                save_path=os.path.join(output_dir, "after_scaling.png"),
            )
            with open(os.path.join(output_dir, "metrics.txt"), "w") as f:
                f.write(f"Optimal T: {T_final:.4f}\n")
                f.write(f"Before Temp - NLL: {before_nll:.3f}, ECE: {before_ece:.3f}\n")
                f.write(f"After Temp  - NLL: {after_nll:.3f}, ECE: {after_ece:.3f}\n")

        torch.save(torch.tensor(T_final), save_path)
        print(f"Saved optimal temperature to: {save_path}")

        return T_final, after_nll, after_ece


def calculate_ece(logits, labels, n_bins=15):
    softmaxes = F.softmax(logits, dim=1)
    confidences, predictions = torch.max(softmaxes, 1)
    accuracies = predictions.eq(labels)

    bins = torch.linspace(0, 1, n_bins + 1)
    bin_lowers = bins[:-1]
    bin_uppers = bins[1:]

    ece = 0.0
    for i, (bin_lower, bin_upper) in enumerate(zip(bin_lowers, bin_uppers)):
        # use le for the final bin so confidence == 1.0 is included
        if i == n_bins - 1:
            in_bin = confidences.ge(bin_lower) * confidences.le(bin_upper)
        else:
            in_bin = confidences.ge(bin_lower) * confidences.lt(bin_upper)
        prop_in_bin = in_bin.float().mean()
        if prop_in_bin.item() > 0:
            accuracy_in_bin = accuracies[in_bin].float().mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            ece += torch.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin

    return ece.item()

def remap_layernorm_keys(state_dict):
    new_state = {}
    for k, v in state_dict.items():
        new_k = (k.replace(".LayerNorm.gamma", ".LayerNorm.weight")
                  .replace(".LayerNorm.beta", ".LayerNorm.bias"))
        new_state[new_k] = v
    return new_state

def make_reliability_diagram(logits, labels, n_bins=15, title="Reliability Diagram", save_path=None):
    softmaxes = F.softmax(logits, dim=1)
    confidences, predictions = softmaxes.max(1)
    accuracies = predictions.eq(labels)

    bins = torch.linspace(0, 1, n_bins + 1)
    width = 1.0 / n_bins
    bin_centers = np.linspace(0, 1.0 - width, n_bins) + width / 2

    bin_indices = []
    for i, (bin_lower, bin_upper) in enumerate(zip(bins[:-1], bins[1:])):
        if i == n_bins - 1:
            bin_indices.append(confidences.ge(bin_lower) * confidences.le(bin_upper))
        else:
            bin_indices.append(confidences.ge(bin_lower) * confidences.lt(bin_upper))

    bin_corrects = np.array([torch.mean(accuracies[bin_idx].float()).item() if bin_idx.any() else 0 for bin_idx in bin_indices])
    bin_scores = np.array([torch.mean(confidences[bin_idx].float()).item() if bin_idx.any() else 0 for bin_idx in bin_indices])

    plt.figure(figsize=(8, 8))
    gap = (bin_scores - bin_corrects)
    plt.bar(bin_centers, bin_corrects, width=width, alpha=0.5, ec='black', label='Outputs')
    plt.bar(bin_centers, gap, bottom=bin_corrects, color='red', alpha=0.5, width=width, hatch='//', edgecolor='r', label='Gap')
    plt.plot([0, 1], [0, 1], '--', color='gray')
    plt.legend(loc='best', fontsize='small')

    ece = calculate_ece(logits, labels, n_bins)
    bbox_props = dict(boxstyle="round", fc="lightgrey", ec="brown", lw=2)
    plt.text(0.2, 0.85, f"ECE: {ece:.2f}", ha="center", va="center", size=20, weight='bold', bbox=bbox_props)

    plt.title(title, size=20)
    plt.ylabel("Accuracy (P[y])", size=18)
    plt.xlabel("Confidence", size=18)
    plt.xlim(0, 1)
    plt.ylim(0, 1)

    if save_path:
        plt.savefig(save_path, dpi=600, bbox_inches="tight")
    else:
        plt.show()
    plt.close()

import glob

def find_checkpoint(model_dir):
    ckpts = glob.glob(os.path.join(model_dir, "checkpoint-*"))
    if not ckpts:
        raise FileNotFoundError(f"No checkpoint-* folder found in {model_dir}")
    # pick the highest step number (last checkpoint)
    return max(ckpts, key=lambda p: int(p.rstrip("/").split("-")[-1]))

In [ ]:
import re
import json
from safetensors.torch import load_file
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
device = "cuda:0" if torch.cuda.is_available() else "cpu"

results_summary = []

for N in range(40, 100, 10):   # 10, 20, 30, ..., 90
    print(f"\n{'='*20} Cycle group: {N} {'='*20}")

    model_dir  = f"/content/drive/MyDrive/bgf_v2/models/bgf_150M_{N}_1ep"
    model_path = find_checkpoint(model_dir)
    print(f"Using checkpoint: {model_path}")
    validation_csv = f"/content/drive/MyDrive/bgf_v2/data/val/bgf_val_{N}.csv"
    json_path      = f"/content/drive/MyDrive/bgf_v2/data/cyc_id_{N}_label_map.json"
    base_temp_dir  = f"/content/drive/MyDrive/bgf_v2/temperature_scaling/bgf_{N}_init1.0"

    try:
        with open(json_path, 'r') as f:
            label_mapper = json.load(f)
        print(f"Label mapper sample: {list(label_mapper.items())[:3]}")

        config = ESMConfig.from_pretrained(model_path)
        state = load_file(os.path.join(model_path, "model.safetensors"))
        state = remap_layernorm_keys(state)

        n_keep = max(int(m.group(1)) for k in state
                     if (m := re.search(r"encoder\.layer\.(\d+)\.", k))) + 1

        model = ESM2LoRAForSequenceClassification(config)
        base = model.get_submodule("backbone.base_model.model")
        base.encoder.layer = nn.ModuleList(list(base.encoder.layer)[:n_keep])
        base.pooler = None
        model.layer_idx = 15

        inc = model.load_state_dict(state, strict=False)
        assert not inc.missing_keys and not inc.unexpected_keys, inc
        model.eval()
        print(f"clean load: {n_keep} layers, no pooler")

        model_with_temp = ModelWithTemperature(model).to(device)
        model_with_temp.eval()

        val_dataset = TokenizedDataset(validation_csv, tokenizer, label_mapper)
        collator = DataCollatorWithPadding(tokenizer=tokenizer, padding='longest', return_tensors='pt')
        val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collator)

        # --- collect logits ---
        print("Collecting logits and labels (one-time pass)...")
        logits_list, labels_list = [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Validation pass [{N}]", unit="batch"):
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels         = batch['labels'].to(device)
                _, logits = model_with_temp.model(
                    input_ids=input_ids, attention_mask=attention_mask
                )
                logits_list.append(logits.float().cpu())
                labels_list.append(labels.cpu())
        all_logits = torch.cat(logits_list, dim=0)
        all_labels = torch.cat(labels_list, dim=0)

        full_acc = (all_logits.argmax(-1) == all_labels).float().mean().item()
        print(f"Full validation accuracy: {full_acc:.4f}")

        # --- manual T sweep ---
        print("\n=== Manual T sweep ===")
        print(f"{'T':>6} {'ECE':>8} {'mean_conf':>10} {'acc':>8}")
        manual_sweep_results = []
        for T in [0.5, 0.7, 1.0, 1.2, 1.5, 2.0, 3.0, 5.0]:
            scaled = all_logits / T
            ece = calculate_ece(scaled, all_labels)
            acc = (scaled.argmax(1) == all_labels).float().mean().item()
            conf = F.softmax(scaled, dim=1).max(1).values.mean().item()
            print(f"{T:>6.2f} {ece:>8.3f} {conf:>10.3f} {acc:>8.3f}")
            manual_sweep_results.append((T, ece, conf, acc))

        os.makedirs(base_temp_dir, exist_ok=True)
        with open(os.path.join(base_temp_dir, "manual_T_sweep.txt"), "w") as f:
            f.write(f"{'T':>6} {'ECE':>8} {'mean_conf':>10} {'acc':>8}\n")
            for T, ece, conf, acc in manual_sweep_results:
                f.write(f"{T:>6.2f} {ece:>8.3f} {conf:>10.3f} {acc:>8.3f}\n")

        # --- baseline ---
        nll_criterion = nn.CrossEntropyLoss()
        before_nll = nll_criterion(all_logits, all_labels).item()
        before_ece = calculate_ece(all_logits, all_labels)
        make_reliability_diagram(
            all_logits, all_labels,
            title=f"Before Temperature Scaling (N={N})\nECE: {before_ece:.3f}",
            save_path=os.path.join(base_temp_dir, "before_scaling.png"),
        )
        print(f"Pre-scaling baseline - NLL: {before_nll:.3f}, ECE: {before_ece:.3f}")

        output_dir = os.path.join(base_temp_dir, "fit")
        os.makedirs(output_dir, exist_ok=True)
        temp_save_path = os.path.join(output_dir, "optimal_temperature.pt")

        model_with_temp.reset_temperature(init=1.0)
        T_opt, after_nll, after_ece = model_with_temp.fit_temperature(
            all_logits, all_labels,
            save_path=temp_save_path, device=device,
            output_dir=output_dir,
            before_nll=before_nll, before_ece=before_ece,
        )

        print("\n=== Fit summary ===")
        print(f"{'T':>8} {'NLL':>8} {'ECE':>8}")
        print(f"{T_opt:>8.3f} {after_nll:>8.3f} {after_ece:>8.3f}")
        with open(os.path.join(base_temp_dir, "fit_summary.txt"), "w") as f:
            f.write(f"Validation accuracy: {full_acc:.4f}\n")
            f.write(f"Baseline NLL: {before_nll:.4f}, ECE: {before_ece:.4f}\n\n")
            f.write(f"{'T':>8} {'NLL':>8} {'ECE':>8}\n")
            f.write(f"{T_opt:>8.4f} {after_nll:>8.4f} {after_ece:>8.4f}\n")

        results_summary.append({
            "N": N, "acc": full_acc,
            "before_nll": before_nll, "before_ece": before_ece,
            "T": T_opt, "after_nll": after_nll, "after_ece": after_ece,
        })

    except Exception as e:
        print(f"!!! Failed for N={N}: {e}")
        results_summary.append({"N": N, "error": str(e)})
        continue

# --- combined summary across all N ---
print("\n\n" + "="*60)
print("COMBINED SUMMARY — all cycle groups")
print("="*60)
for r in results_summary:
    if "error" in r:
        print(f"N={r['N']:>3}: ERROR - {r['error']}")
    else:
        print(f"N={r['N']:>3}  acc={r['acc']:.4f}  T={r['T']:.3f}  "
              f"before_ece={r['before_ece']:.3f}  after_ece={r['after_ece']:.3f}")

NameError: name 'torch' is not defined